# Cuarta Iteración: Arquitectura Avanzada en PyTorch (Gating Network + Focal Loss)

### Los 2 Avances Científicos Implementados Aquí:

**1. Mecanismo de Apagado Dinámico (Gating / Atención):**
Hemos construido una red neuronal personalizada que tiene un "Juez". Este juez analiza las variables originales del cliente y emite un voto de confianza (entre 0 y 1) para LightGBM y CatBoost. Si el Juez detecta que el cliente pertenece a un segmento donde los árboles suelen fallar, multiplica la predicción del árbol por 0.01 (apagándola casi por completo) y le da toda la voz a la Red Neuronal profunda.

**2. Aproximación a Optimización de AUC (Focal Loss):**
El AUC es una métrica de "Rankeo" (función escalonada) y por tanto no tiene derivada matemática. Para lograr que la red aprenda algo similar al AUC (que priorice la minoría difícil y margine a la mayoría fácil), reemplazamos la pérdida clásica por **Focal Loss**. Focal Loss aniquila matemáticamente la influencia de los clientes fáciles (el 85% de clase 0) y concentra casi todo el descenso de gradiente en los clientes difíciles que los árboles no pudieron predecir bien.

In [ ]:
import os
# 🛡️ PARCHE ANTI-DEADLOCK PARA MAC (APPLE SILICON)
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import warnings
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
print("✅ Librerías de PyTorch e IA Clásica importadas.")

In [ ]:
# 1. CARGA Y PARTICIÓN DEL DATASET (Igual que en la Fase 3)
train_df = pd.read_csv('../../dataset/train.csv')

X = train_df.drop(['id_cliente', 'objetivo'], axis=1)
y = train_df['objetivo']

X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=42
)

cat_cols = ['ocupacion', 'region', 'canal_adquisicion', 'banda_riesgo', 'dispositivo_principal']
num_cols = [c for c in X.columns if c not in cat_cols]

X_train_cb = X_train_full.copy()
X_test_cb = X_test_final.copy()
for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)
    X_train_full[col] = X_train_full[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

print("✅ Datos divididos y categorías ajustadas.")

In [ ]:
# 2. GENERACIÓN DE PREDICCIONES OOF DE ÁRBOLES EXPERTOS
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_train_full))
test_preds_lgb = np.zeros(len(X_test_final))
oof_cb = np.zeros(len(X_train_full))
test_preds_cb = np.zeros(len(X_test_final))

best_lgb_params = {'num_leaves': 42, 'learning_rate': 0.017766463699737917, 'min_child_samples': 44, 'subsample': 0.6548629928035845, 'colsample_bytree': 0.595372408030665, 'reg_alpha': 4.9893861303564696e-05, 'reg_lambda': 0.1440126277634314}

print("🌳 Entrenando Expertos Base (LightGBM y CatBoost)...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    # LightGBM
    model_lgb = lgb.LGBMClassifier(objective='binary', metric='auc', n_estimators=1000, random_state=42, verbose=-1, **best_lgb_params)
    model_lgb.fit(X_train_full.iloc[train_idx], y_train_full.iloc[train_idx], 
                  eval_set=[(X_train_full.iloc[val_idx], y_train_full.iloc[val_idx])], callbacks=[lgb.early_stopping(30, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict_proba(X_train_full.iloc[val_idx])[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test_final)[:, 1] / skf.n_splits
    
    # CatBoost
    model_cb = CatBoostClassifier(eval_metric='AUC', random_seed=42, early_stopping_rounds=30, verbose=False, thread_count=-1)
    train_pool = Pool(X_train_cb.iloc[train_idx], y_train_full.iloc[train_idx], cat_features=cat_cols)
    val_pool = Pool(X_train_cb.iloc[val_idx], y_train_full.iloc[val_idx], cat_features=cat_cols)
    model_cb.fit(train_pool, eval_set=val_pool, use_best_model=True)
    oof_cb[val_idx] = model_cb.predict_proba(val_pool)[:, 1]
    test_preds_cb += model_cb.predict_proba(Pool(X_test_cb, cat_features=cat_cols))[:, 1] / skf.n_splits

print(f"✅ OOF LightGBM AUC: {roc_auc_score(y_train_full, oof_lgb):.5f}")
print(f"✅ OOF CatBoost AUC: {roc_auc_score(y_train_full, oof_cb):.5f}")

In [ ]:
# 3. PREPARACIÓN DE TENSORES PARA PYTORCH
import time
print("⚙️ Escalando datos para PyTorch...")
t0 = time.time()
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)
print("ColumnTransformer instanciado.")

X_train_rn_base = preprocessor.fit_transform(X_train_full)
print(f"fit_transform terminado en {time.time()-t0:.2f}s")
X_test_rn_base = preprocessor.transform(X_test_final)
print(f"transform test terminado en {time.time()-t0:.2f}s")

train_tree_preds = np.column_stack([oof_lgb, oof_cb])
test_tree_preds = np.column_stack([test_preds_lgb, test_preds_cb])

X_t_feat, X_v_feat, X_t_preds, X_v_preds, y_t, y_v = train_test_split(
    X_train_rn_base, train_tree_preds, y_train_full.values, test_size=0.15, stratify=y_train_full.values, random_state=42
)
print(f"train_test_split terminado en {time.time()-t0:.2f}s")

tensor_X_t_feat = torch.tensor(X_t_feat, dtype=torch.float32)
tensor_X_t_preds = torch.tensor(X_t_preds, dtype=torch.float32)
tensor_y_t = torch.tensor(y_t, dtype=torch.float32).view(-1, 1)

tensor_X_v_feat = torch.tensor(X_v_feat, dtype=torch.float32)
tensor_X_v_preds = torch.tensor(X_v_preds, dtype=torch.float32)
tensor_y_v = torch.tensor(y_v, dtype=torch.float32).view(-1, 1)

tensor_X_test_feat = torch.tensor(X_test_rn_base, dtype=torch.float32)
tensor_X_test_preds = torch.tensor(test_tree_preds, dtype=torch.float32)
print(f"✅ Tensores generados en {time.time()-t0:.2f}s")

In [ ]:
# 4. LA ARQUITECTURA DE IA AVANZADA (Gating + Focal Loss)
class DynamicGatingMetaLearner(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        
        # 1. El "Juez": Decide cuánto confiar en LightGBM y CatBoost
        self.gating_network = nn.Sequential(
            nn.Linear(num_features, 32),
            nn.ReLU(),
            nn.Linear(32, 2), # 2 salidas (pesos de confianza para los 2 árboles)
            nn.Sigmoid() # Fuerza a que la confianza sea entre 0 y 1
        )
        
        # 2. La Red Principal: Analiza el cliente a profundidad
        self.main_network = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # 3. Mezclador Final: Junta la opinión de los árboles (ya modulada) + opinión propia
        self.final_mixer = nn.Linear(3, 1)
        
    def forward(self, x_features, tree_preds):
        # El Juez evalúa las variables y emite sus votos de confianza
        gating_weights = self.gating_network(x_features) 
        
        # Apagado Dinámico (Atención): Multiplica predicciones de los árboles por la confianza
        gated_tree_preds = tree_preds * gating_weights
        
        # La Red emite su propia probabilidad
        nn_pred = self.main_network(x_features)
        
        # Mezcla final
        combined = torch.cat([gated_tree_preds, nn_pred], dim=1)
        return torch.sigmoid(self.final_mixer(combined))

# Implementación de Focal Loss (Aproximación de Ranking/AUC)
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss) 
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss
        return torch.mean(focal_loss)


In [ ]:
# 5. BUCLE DE ENTRENAMIENTO PERSONALIZADO CON EARLY STOPPING POR AUC
num_features = tensor_X_t_feat.shape[1]
model = DynamicGatingMetaLearner(num_features)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = FocalLoss(gamma=2.0) # Forzamos a la red a enfocarse en los casos difíciles

epochs = 300
batch_size = 512
best_auc = 0
patience = 20
patience_counter = 0
best_model_weights = None

print("🚀 Iniciando Entrenamiento PyTorch (Focal Loss + Gating + AUC Early Stopping)")
for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(tensor_X_t_feat.size()[0])
    
    # Mini-batches
    for i in range(0, tensor_X_t_feat.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_x_feat, batch_x_preds, batch_y = tensor_X_t_feat[indices], tensor_X_t_preds[indices], tensor_y_t[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_x_feat, batch_x_preds)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
    # Evaluación por Época
    model.eval()
    with torch.no_grad():
        val_outputs = model(tensor_X_v_feat, tensor_X_v_preds)
        val_auc = roc_auc_score(tensor_y_v.numpy(), val_outputs.numpy())
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_model_weights = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            
    if (epoch+1) % 10 == 0:
        print(f"Época {epoch+1:03d} | AUC Validación Limpia: {val_auc:.5f} | Mejor AUC Histórico: {best_auc:.5f}")
        
    if patience_counter >= patience:
        print(f"\n🛑 Early Stopping en la época {epoch+1}. Restaurando el mejor cerebro (AUC={best_auc:.5f})")
        model.load_state_dict(best_model_weights)
        break

In [ ]:
# 6. EVALUACIÓN FINAL SOBRE EL 10% TEST PURO
model.eval()
with torch.no_grad():
    final_probs = model(tensor_X_test_feat, tensor_X_test_preds).numpy()

# Como el AUC no depende del threshold, usamos el threshold óptimo para F1 si queremos matriz de confusión balanceada
# Por simplicidad usamos 0.5 o el promedio de las probabilidades.
threshold = np.percentile(final_probs, 85) # Ajustamos umbral al top 15% (distribución real de la clase 1)
final_preds_bin = (final_probs >= threshold).astype(int)

final_auc = roc_auc_score(y_test_final, final_probs)
final_gini = 2.0 * final_auc - 1.0

print("\n=======================================================")
print("📊 RESULTADOS FINALES SOBRE EL SET DE PRUEBA (10% PURO)")
print("=======================================================")
print(f"⭐ AUC FINAL:  {final_auc:.5f}")
print(f"🏆 GINI FINAL: {final_gini:.5f}\n")
print("📉 MATRIZ DE CONFUSIÓN (Threshold Ajustado):")
print(confusion_matrix(y_test_final, final_preds_bin))
print("\n📋 REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test_final, final_preds_bin))